In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

encoded_path = Path("/content/hr_employee_attrition_encoded.csv")
df = pd.read_csv(encoded_path)

if "Attrition_Flag" not in df.columns:
    if "Attrition" in df.columns:
        df["Attrition_Flag"] = df["Attrition"].map({"Yes": 1, "No": 0})
    else:
        raise ValueError("Attrition_Flag or Attrition column is required for modeling")

target_col = "Attrition_Flag"
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols]
y = df[target_col]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X_numeric = X[numeric_cols]

sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_numeric, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=200)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

print("Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr))
print("Logistic Regression Confusion Matrix")
print(confusion_matrix(y_test, y_pred_lr))

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf))
print("Random Forest Confusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

feature_importances = pd.DataFrame(
    {
        "feature": X_train.columns,
        "importance": rf.feature_importances_
    }
).sort_values("importance", ascending=False)

output_dir = Path("reports")
output_dir.mkdir(parents=True, exist_ok=True)
feature_importances.to_csv(output_dir / "feature_importances_random_forest.csv", index=False)


Logistic Regression Classification Report
              precision    recall  f1-score   support

           0       0.81      0.80      0.80       247
           1       0.80      0.81      0.81       247

    accuracy                           0.81       494
   macro avg       0.81      0.81      0.81       494
weighted avg       0.81      0.81      0.81       494

Logistic Regression Confusion Matrix
[[198  49]
 [ 47 200]]
Random Forest Classification Report
              precision    recall  f1-score   support

           0       0.92      0.89      0.91       247
           1       0.89      0.93      0.91       247

    accuracy                           0.91       494
   macro avg       0.91      0.91      0.91       494
weighted avg       0.91      0.91      0.91       494

Random Forest Confusion Matrix
[[220  27]
 [ 18 229]]
